# Object size and concept quality: crop vs. circle (VOC, COCO)

In [ ]:
#Setup: subset, synonyms, buckets, GT objects 
import sys, os, json, time, pickle
ROOT = "/workspaces/Object-Centric-Concept-Bottlenecks"
sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import h5py
import xml.etree.ElementTree as ET
from collections import Counter
from data import PascalVOCDataset, CocoDataset

from eval_utils import PATHS, METHODS, CLASS_NAMES, COCO_CLASSES, vocab_idx, SYNONYMS, draw_subset, rank_concepts

IMG_SIZE           = 224         
AREA_MIN, AREA_MAX = 0.01, 0.85  # = pipeline filter 
BUCKETS = {
    "voc":  [(0.01, 0.05), (0.05, 0.10), (0.10, 0.20), (0.20, 0.30), (0.30, 0.85)],
    "coco": [(0.01, 0.03), (0.03, 0.05), (0.05, 0.10), (0.10, 0.20), (0.20, 0.85)],
}

def bucket_name(lo, hi):
    return f"{lo * 100:.0f}–{hi * 100:.0f} %"

def assign_bucket(area, ds):
    for i, (lo, hi) in enumerate(BUCKETS[ds]):
        if lo <= area < hi or (i == len(BUCKETS[ds]) - 1 and area == hi):
            return bucket_name(lo, hi)
    return None

voc_val  = PascalVOCDataset(f"{ROOT}/data/voc", year="2012", image_set="val", download=False)
coco_val = CocoDataset(f"{ROOT}/datasets/coco", train_val="val")
coco_idx = {int(fn.split(".")[0]): i for i, fn in enumerate(coco_val.image_files)}

with open(f"{ROOT}/datasets/coco/annotations/instances_val2017.json") as f:
    _coco = json.load(f)
COCO_SIZE = {im["id"]: (im["width"], im["height"]) for im in _coco["images"]}
COCO_ANNS = {}
for a in _coco["annotations"]:
    COCO_ANNS.setdefault(a["image_id"], []).append(a)


def gt_objects(ds, iid):
    """All GT objects of an image in original coordinates (x1, y1, x2, y2); 'drop' = difficult/iscrowd."""
    if ds == "voc":
        root = ET.parse(voc_val.annotations[int(iid)]).getroot()
        size = int(root.find("size/width").text), int(root.find("size/height").text)
        objs = []
        for o in root.findall("object"):
            bb, d = o.find("bndbox"), o.find("difficult")
            objs.append({"cls": o.find("name").text,
                         "box": [float(bb.find(t).text) for t in ("xmin", "ymin", "xmax", "ymax")],
                         "drop": "difficult" if d is not None and int(d.text) else None})
    else:
        size = COCO_SIZE[int(iid)]
        objs = [{"cls": COCO_CLASSES[a["category_id"] - 1],
                 "box": [a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]],
                 "drop": "iscrowd" if a["iscrowd"] else None}
                for a in COCO_ANNS.get(int(iid), [])]
    return size, objs

print("Setup OK")

In [ ]:
#Collect and filter GT objects of the subset, count per bucket
def collect_objects(ds):
    rows, labels, _, _ = draw_subset(ds)
    with h5py.File(PATHS[ds]["full"], "r") as f:
        img_ids = f["test_img_ids"][:].ravel()[rows]
    names = CLASS_NAMES[ds]
    col_of = {n: c for c, n in enumerate(names) if n is not None}
    recs, label_ok = [], []
    for k, iid in enumerate(img_ids):
        (W, H), objs = gt_objects(ds, iid)
        label_ok.append({names[c] for c in np.flatnonzero(labels[k])}
                        == {o["cls"] for o in objs if ds == "voc" or o["drop"] is None})
        n_cls = Counter(o["cls"] for o in objs)   
        sx, sy = IMG_SIZE / W, IMG_SIZE / H
        for j, o in enumerate(objs):
            x1, y1, x2, y2 = o["box"]
            box = [x1 * sx, y1 * sy, x2 * sx, y2 * sy]
            area = (box[2] - box[0]) * (box[3] - box[1]) / IMG_SIZE ** 2
            if o["drop"]:
                reason = o["drop"]
            elif not AREA_MIN <= area <= AREA_MAX:
                reason = "area"
            elif not SYNONYMS[ds][o["cls"]]:
                reason = "no synonym"
            else:
                reason = None
            recs.append({"k": k, "img_id": iid, "obj": j, "cls": o["cls"], "col": col_of[o["cls"]],
                         "box224": box, "area": area, "n_same_cls": n_cls[o["cls"]], "excluded": reason})
    return pd.DataFrame(recs), rows, img_ids, np.array(label_ok)


OBJ, IMG = {}, {}
count_rows = []
for ds in ("voc", "coco"):
    df, _, IMG[ds], label_ok = collect_objects(ds)
    OBJ[ds] = df[df["excluded"].isna()].reset_index(drop=True)
    OBJ[ds]["bucket"] = [assign_bucket(a, ds) for a in OBJ[ds]["area"]]
    assert OBJ[ds]["bucket"].notna().all()
    kept = OBJ[ds]
    print(f"{ds.upper()}: images {len(IMG[ds])} | box classes = .h5 labels: {label_ok.all()} ({(~label_ok).sum()} mismatches) "
          f"| GT objects {len(df)} | excluded: {df['excluded'].value_counts().to_dict()}")
    no_syn = df.loc[df["excluded"] == "no synonym", "cls"].value_counts().to_dict()
    print(f"      excluded without synonym: {no_syn or '–'}")
    print(f"      evaluated: {len(kept)} objects in {kept['k'].nunique()} images "
          f"(mean {len(kept) / kept['k'].nunique():.2f}/image) | of these, class unique in image: {(kept['n_same_cls'] == 1).sum()}")
    for lo, hi in BUCKETS[ds]:
        g = kept[kept["bucket"] == bucket_name(lo, hi)]
        count_rows.append({"Dataset": ds.upper(), "Bucket": bucket_name(lo, hi),
                           "Objects": len(g), "Images": g["k"].nunique(),
                           "Objects (class unique)": int((g["n_same_cls"] == 1).sum())})

bucket_counts = pd.DataFrame(count_rows).set_index(["Dataset", "Bucket"])
display(bucket_counts)

In [ ]:
import torch
import torchvision.transforms.v2 as T
from PIL import Image
from scipy import sparse
from tqdm import tqdm
import splice
from utils import gather_object_embeddings

DEVICE = "cuda"

# SpLiCE and preprocessing exactly as in generate_concepts.py
splice_pipe = splice.load("open_clip:ViT-B-32", "laion", 10000, l1_penalty=0.25, device=DEVICE, return_weights=True)
custom_preprocessor = T.Compose([
    T.Resize(size=224, interpolation=T.InterpolationMode.BICUBIC, max_size=None, antialias=True),
    T.CenterCrop(size=(224, 224)),
    T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)),
])


def load_224(ds, iid):
    if ds == "voc":
        img, _, _, _ = voc_val[int(iid)]
        path = voc_val.images[int(iid)]
    else:
        img, _, _, _ = coco_val[coco_idx[int(iid)]]
        path = os.path.join(coco_val.image_dir, coco_val.image_files[coco_idx[int(iid)]])
    assert Image.open(path).size == gt_objects(ds, iid)[0], f"image size ≠ annotation: {path}"
    return img


def encode_objects(image, boxes, circle):
    n = len(boxes)
    emb, used, n_used = gather_object_embeddings(
        boxes, torch.ones(n), 0.0, n, image, DEVICE,
        splice_pipe, 10000, custom_preprocessor, draw_boxes=False, draw_circles=circle,
    )
    assert n_used == n and torch.equal(used, boxes)
    return emb.cpu().numpy()

print("Models loaded")

In [ ]:
#Encodings for all GT objects 
CACHE = f"{ROOT}/red_circle_notebooks/bucket_cache_{{ds}}.pkl"

def compute_encodings(ds):
    df = OBJ[ds]
    assert df["k"].is_monotonic_increasing
    out = {"key": df[["img_id", "obj"]].to_numpy(), "crop": [], "circle": []}
    groups = list(df.groupby("k", sort=True))  
    t0 = time.time()
    for n_done, (k, g) in enumerate(tqdm(groups, desc=ds.upper()), 1):
        image = load_224(ds, IMG[ds][k]).to(DEVICE)
        boxes = torch.tensor(np.array(g["box224"].tolist()), dtype=torch.float32)
        with torch.no_grad():
            for m in METHODS:
                out[m].append(sparse.csr_matrix(encode_objects(image, boxes, circle=(m == "circle"))))
        if n_done == 20:
            dt = (time.time() - t0) / n_done
            print(f"  {dt:.2f} s/image → estimated {dt * len(groups) / 60:.0f} min for {ds.upper()}")
    for m in METHODS:
        out[m] = sparse.vstack(out[m]).tocsr()
    return out


ENC = {}
for ds in ("voc", "coco"):
    cache = CACHE.format(ds=ds)
    if os.path.exists(cache):
        with open(cache, "rb") as fh:
            ENC[ds] = pickle.load(fh)
        print(f"{ds.upper()}: cache loaded ({cache})")
    else:
        ENC[ds] = compute_encodings(ds)
        with open(cache, "wb") as fh:
            pickle.dump(ENC[ds], fh)
    assert np.array_equal(ENC[ds]["key"], OBJ[ds][["img_id", "obj"]].to_numpy()), "cache does not match OBJ"
    assert all(ENC[ds][m].shape[0] == len(OBJ[ds]) for m in METHODS)
    assert min(ENC[ds][m].min() for m in METHODS) >= 0, "negative SpLiCE weights – check definition of active"
    print(f"{ds.upper()}: {len(OBJ[ds])} objects | mean active concepts " +
          ", ".join(f"{m} {ENC[ds][m].getnnz(axis=1).mean():.1f}" for m in METHODS))

In [ ]:
# Metrics per object (own concept vector) and tables per bucket 
def object_metrics(emb, cols, ds):
    """Found: at least one synonym active (> 0). Recall@10: a synonym among the top 10 active concepts."""
    names, syn = CLASS_NAMES[ds], SYNONYMS[ds]
    label_sets = {col: np.array([vocab_idx[w] for w in syn[n]]) for col, n in enumerate(names) if n is not None}
    found, r10 = [], []
    for i, c in enumerate(cols):
        v = emb[i].toarray().ravel()
        ranking = rank_concepts(v, v, "sum")   
        found.append(bool((v[label_sets[c]] > 0).any()))
        r10.append(bool(np.isin(label_sets[c], ranking[:10]).any()))
    return np.array(found), np.array(r10)


def bucket_table(df, ds):
    groups = [(bucket_name(lo, hi), df[df["bucket"] == bucket_name(lo, hi)]) for lo, hi in BUCKETS[ds]] + [("Total", df)]
    rows = []
    for name, g in groups:
        r = {"Dataset": ds.upper(), "Bucket": name, "Objects": len(g), "Images": g["k"].nunique()}
        for met in ("Found", "Recall@10"):
            crop, circle = g[f"{met}_crop"].to_numpy(float), g[f"{met}_circle"].to_numpy(float)
            r[f"{met} Crop"], r[f"{met} Circle"] = crop.mean(), circle.mean()
            r[f"Δ {met}"] = crop.mean() - circle.mean()
        rows.append(r)
    return rows


rows_all, rows_unique = [], []
for ds in ("voc", "coco"):
    df = OBJ[ds].copy()
    df["bucket"] = [assign_bucket(a, ds) for a in df["area"]]
    for m in METHODS:
        df[f"Found_{m}"], df[f"Recall@10_{m}"] = object_metrics(ENC[ds][m], df["col"].to_numpy(), ds)
    OBJ[ds] = df
    rows_all    += bucket_table(df, ds)
    rows_unique += bucket_table(df[df["n_same_cls"] == 1], ds)

bucket_results        = pd.DataFrame(rows_all).set_index(["Dataset", "Bucket"])
bucket_results_unique = pd.DataFrame(rows_unique).set_index(["Dataset", "Bucket"])
print("All objects (Δ = crop − circle):")
display(bucket_results.round(3))
print("Only objects whose class is annotated exactly once in the image:")
display(bucket_results_unique.round(3))

Size figure for the thesis (all objects vs. category occurs once).

In [ ]:
# Size figure: Found (object-level recall), crop vs. circle
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

EXPECTED = {
    "all": {"VOC":  [(0.317, 0.251), (0.433, 0.278), (0.592, 0.428), (0.579, 0.502), (0.749, 0.708)],
            "COCO": [(0.189, 0.179), (0.293, 0.228), (0.413, 0.287), (0.464, 0.350), (0.534, 0.505)]},
    "once": {"VOC":  [(0.479, 0.121), (0.530, 0.272), (0.689, 0.385), (0.650, 0.505), (0.795, 0.762)],
             "COCO": [(0.195, 0.081), (0.353, 0.140), (0.490, 0.192), (0.539, 0.288), (0.568, 0.515)]},
}
ROWS = {"all": (bucket_results, "All objects"), "once": (bucket_results_unique, "Category occurs once")}
for key, (res, _) in ROWS.items():
    for ds, exp in EXPECTED[key].items():
        got = res.loc[ds].drop("Total")[["Found Crop", "Found Circle"]].round(3).to_numpy()
        assert np.allclose(got, exp, atol=1e-9), f"{key}/{ds}: values do not match\n{got}"
print("Check OK: all values match the expected ones")

COLORS = {"Crop": "#2A78D6", "Circle": "#EB6834"}
MARKERS = {"Crop": "o", "Circle": "s"}
CM = 1 / 2.54

sns.set_theme(context="paper", style="ticks", rc={
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 9,
    "axes.edgecolor": "#52514E", "xtick.color": "#52514E", "ytick.color": "#52514E",
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

fig, axes = plt.subplots(2, 2, figsize=(14.1 * CM, 10 * CM), sharex="col", sharey=True, layout="constrained")
for r, (key, (res, row_title)) in enumerate(ROWS.items()):
    for c, ds in enumerate(("VOC", "COCO")):
        ax = axes[r, c]
        t = res.loc[ds].drop("Total")
        x = np.arange(len(t))
        for m in ("Crop", "Circle"):
            ax.plot(x, t[f"Found {m}"], color=COLORS[m], marker=MARKERS[m], markersize=4.5, linewidth=1.5, label=m)
        ax.set_xticks(x, [b.replace(" %", "%") for b in t.index])
        ax.set_ylim(0, 1)
        ax.set_yticks(np.arange(0, 1.01, 0.2))
        ax.yaxis.grid(True, color="#E4E3DF", linewidth=0.6)
        ax.set_axisbelow(True)
        # Panel letter and N in the top left corner
        letter = "abcd"[2 * r + c]
        ax.text(0.03, 0.97, f"({letter})", transform=ax.transAxes, ha="left", va="top", fontweight="bold")
        ax.text(0.15, 0.97, f"N = {res.loc[(ds, 'Total'), 'Objects']:,}", transform=ax.transAxes,
                ha="left", va="top", fontsize=8, color="#6E6D69")
        if r == 0:
            ax.set_title(ds, fontsize=10)
        if c == 0:
            ax.set_ylabel(row_title, fontsize=9)
        sns.despine(ax=ax)

fig.supylabel("Object-level recall", fontsize=9)
fig.supxlabel("Relative box area", fontsize=9)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False)

FIG_DIR = f"{ROOT}/red_circle_notebooks/figures"
os.makedirs(FIG_DIR, exist_ok=True)
fig.savefig(f"{FIG_DIR}/size_buckets.pdf", bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/size_buckets.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR}/size_buckets.pdf (+ .png)")